# LLMs

Contents:


- How to Use LLMs
  - Local LLM Inference - CPU & GPU
  - Inference via external API
- Key takeaways about LLM through examples
  - Hallucinations
  - LLM Strengths
  - LLM Weaknesses

В качестве API используется локально развернутая модель с использованием ollama

## 0. Environment setup

Google Colab can run notebooks on CPU or GPU. We will demonstrate both options.

Go to Runtime → Change runtime type → GPU

In [1]:
import os, shutil

# Core libs (HF + OpenAI)
# !pip -q install -U huggingface_hub transformers accelerate sentencepiece openai

# llama-cpp-python:
# - CPU: pip installs a prebuilt wheel (fast)
# - GPU (CUDA): needs a local build with GGML_CUDA enabled
gpu_available = shutil.which("nvidia-smi") is not None
print("GPU runtime detected:", gpu_available)

if gpu_available:
    !nvidia-smi
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    os.environ["FORCE_CMAKE"] = "1"
    # Force a rebuild with CUDA support
    # !pip -q install --no-cache-dir --force-reinstall llama-cpp-python

# !pip -q install -U llama-cpp-python


GPU runtime detected: False


## 1. How to Use LLMs

Two common ways to use LLMs in practice:

1. **Local inference with open-weight models**  
   (small models on CPU or larger models on GPU)

2. **Hosted APIs from model providers:**
   - Yandex Cloud AI Studio  
   - OpenAI  
   - ...

### Trade-offs

- **Local inference →** privacy, full control, predictable infrastructure  
  *(but you are responsible for setup and performance)*

- **API →** fastest path to high quality and scalability  
  *(but you pay per token and depend on a provider)*


---

## 1.1 Local LLM Inference

- **Open weights →** you download a model (e.g., from Hugging Face) and run it locally

- **CPU vs GPU**
  - **CPU →** suitable for very small models (demos, simple tasks, prototyping)
  - **GPU →** enables larger models with much better latency and quality

We will start with a tiny CPU-friendly model, then run a GPU example (if a GPU runtime is enabled).


In [10]:
# Set parameters and prompts that we will use through notebook

TEMPERATURE = 0.5
MAX_TOKENS = 400
N_CTX = 2048
SYSTEM_PROMPT = "You're my personal assistant. Always start your response by saying 'Hello, Alyona!'"
USER_QUERY = "Give me 3 ideas where AI agents are useful."

PLAIN_PROMPT = f"""\
  System: {SYSTEM_PROMPT}\n
  User: {USER_QUERY}\n
  Assistant:
"""

MESSAGES_PROMPT = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUERY},
]



#### 1.1.1. CPU demo

In [3]:
# # --- 1) Download a tiny Llama-compatible GGUF quantized model ---
# from huggingface_hub import hf_hub_download

# cpu_model_path = hf_hub_download(
#     repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
#     filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
# )

# print("CPU model path:", cpu_model_path)

In [4]:
# # --- 2) Load with llama.cpp ---
# from llama_cpp import Llama

# cpu_llm = Llama(
#     model_path=cpu_model_path,
#     n_ctx=N_CTX,
#     n_gpu_layers=0, # CPU only
#     verbose=False
# )

# print("✅ llama model loaded for CPU")

In [5]:
# # --- 3) Inference ---

# # --- 3.1) Inference with a single plain prompt ---
# def ask_llm_plain_prompt(llm):
#   out_plain = llm(
#       PLAIN_PROMPT,
#       max_tokens=MAX_TOKENS,
#       temperature=TEMPERATURE,
#   )
#   print("\n--- Plain prompt output ---")
#   print(out_plain["choices"][0]["text"])


# # --- 3.2) Inference with roles via chat completion. That's why roles matter
# def ask_llm_chat_prompt(llm):
#   out_chat = llm.create_chat_completion(
#       messages=MESSAGES_PROMPT,
#       temperature=TEMPERATURE,
#       max_tokens=MAX_TOKENS,
#   )

#   print("\n--- Chat roles output ---")
#   print(out_chat["choices"][0]["message"]["content"])


In [6]:
# ask_llm_plain_prompt(cpu_llm)
# ask_llm_chat_prompt(cpu_llm)

#### 1.1.2. GPU demo


In [7]:
# # --- 1) Download a larger Llama GGUF model ---
# model_path_gpu = hf_hub_download(
#     repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
#     filename="llama-2-7b-chat.Q4_K_M.gguf"
#   )
# print("GPU model path:", model_path_gpu)


# # --- 2) Load with llama.cpp ---

# llm_gpu = Llama(
#     model_path=model_path_gpu,
#     n_ctx=N_CTX,
#     n_gpu_layers=-1,   # offload all layers to GPU
#     verbose=False
# )

# print("🚀 Llama GPU loaded.")


In [8]:
# # --- 3) Inference ---
# ask_llm_plain_prompt(llm_gpu)
# ask_llm_chat_prompt(llm_gpu)

### 1.2. Inference via external API

In this section, we will use the OpenAI API as an example. You will learn how to work with Yandex Cloud AI Studio in the next lessons.

#### 1.2.1 Setup
First, [generate](https://platform.openai.com/api-keys) an API key and save it in Secrets:
- In the left sidebar, click 🔑 Secrets
- Add a new secret:

```
Name: OPENAI_API_KEY
Value: sk-...
```


Or just paste it below.

In [9]:
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#### 1.2.2 Call OpenAI with the same prompts and settings


In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # API-ключ не нужен, но требуется передать любую строку
)
MODEL = "phi3:mini"

def call_openai(messages, model=MODEL):
    resp = client.responses.create(
        model=model,
        input=messages,
    )
    return resp.output_text


print("\n--- OpenAI Chat roles output ---")
print(call_openai(MESSAGES_PROMPT))



--- OpenAI Chat roles output ---
1) Personal Assistants: As an artificial intelligence agent myself, I can understand how helpful it is to have assistance in managing tasks and schedules efficiently on a daily basis - this includes making reminders for important events or meetings, setting alarms, sending quick replies via email, etc.
2) Customer Service Chatbots: AI chatbots like me are designed to provide immediate responses to customer inquiries round the clock without any downtime due to holidays or vacations - whether it's helping a user navigate through an online store menu system or addressing technical issues, I can quickly and accurately respond with relevant information.
3) Virtual Reality Gaming: AI agents are increasingly being used in creating interactive virtual worlds where players feel as if they are actually playing alongside other avatars - this is especially useful for younger audiences who enjoy immersive gaming experiences that blur the lines between digital and p

## 2. Key takeaways about LLM through examples

### 2.1. Hallucinations

The model may confidently generate false information.

In [12]:
def show(title, text):
  print(f"\n=== {title} ===\n{text}\n")

In [14]:
messages = [
    {"role":"system","content":"You are a helpful Python assistant."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages)
show("Hallucination risk", text)


=== Hallucination risk ===
To create an open API with JSON schema definitions using the `client` in python sdk for gRPC, you typically do not directly call methods like `create_json_schema()`. This method doesn't exist as per standard OpenAI Python SDK documentation (as of my last update). However, we can simulate creating a client-side operation to generate and handle JSON schemas using protobuf definitions which are part of the gRPC ecosystem.

Firstly, you need an API definition with appropriate .proto files that define your service structure along with json_schema configurations for each type or request object in this case (since I am unable to create a custom method like `create_json_schema()`, we can show how JSON schema could be generated as part of the normal protobuf processing).

Here's an example setup using OpenAPI definitions (.proto files), which are common with API documentation tools:

```protobuf
syntax = "proto3";

package openapi;

option go_package = "/path/to/your

However, `client.responses.create_json_schema()` does not exist.

Hallucinations can be reduced with clear instructions.

In [15]:
messages = [
    {"role":"system","content":
     "If you are not sure, say 'I don't know'. Do not invent APIs."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages)
show("Cautious mode", text)


=== Cautious mode ===
To create a JSON schema using `client.responses.create_json_schema()` in the Google Cloud's API Client library for Python (also known as "gcloud"), you first need to authenticate your application by setting up credentials, install the gcloud client package if it is not installed yet (`pip install google-api-python-client`), and then write a script that interacts with an appropriate GCP service.

Here's how to create a JSON schema in Python using Google Cloud services:

```python
from googleapiclient.discovery import build
import json
from oauth2client.service_account import ServiceAccountCredentials

# Authenticate and construct the service object
SAMPLE_KEYFILE_CREDENTIALS = 'path/to/your/keyfile.json'  # Replace with your key file path
scoped_user_permission = ('https://www.googleapis.com/' + \
                          client_info['client_id'] + '/auth')
credentials = ServiceAccountCredentials.from_p12_keyfile(SAMPLE_KEYFILE_CREDENTIALS, scoped_user_permission

### 2.2. LLM Strengths

1.  Language tasks such as summarization and paraphring

In [16]:
paragraph = """
Large language models predict the next token rather than verify facts.
They generate fluent text and summaries but may hallucinate.
They struggle with exact counting and up-to-date knowledge without tools.
"""

messages = [
    {"role":"system","content":"Summarize for a lecture slide in a few words"},
    {"role":"user","content":paragraph}
]

text = call_openai(messages)
show("Summarization", text)


=== Summarization ===
"Limits of LLMs in factual accuracy, fluency challenges including 'hallucination,' difficulty in precise quantitative tasks." - Dr. Jane Doe's Perspective on AI Limitations for Today’in Computer Science Courses (Few Words).



2. Generating standard structures

In [17]:
messages = [
    {"role":"system","content":"Return ONLY valid JSON."},
    {"role":"user","content":
     "Generate JSON with keys: strengths (3 items), weaknesses (2 items) of LLM."}
]

text = call_openai(messages)
show("Structured output", text)


=== Structured output ===
{
   "strengths": [
      "Excellent ability to understand and process natural language queries",
      "High accuracy in providing relevant responses based on the input context",
      "Capable of learning from interactions and improving over time"
   ],
   "weaknesses": [
      "Might produce grammatically incorrect sentences or phrases occasionally", 
      "Can generate biased outputs if not properly monitored, due to inherent bias in training data"
   ]
}




### 2.3. LLM Weaknesses

1. Counting characters

In [19]:
s = "a"*37 + "b"*41 + "c"*19 + "b"*5

messages = [
    {"role":"system","content":"Answer with ONE integer only."},
    {"role":"user","content":f"How many characters are in this string?\n{s}"}
]

text = call_openai(messages)
show("Counting characters task", text)

print("Ground truth:", len(s))



=== Counting characters task ===
To determine the number of characters, we count each character once regardless of its repetition. There are:
- 52 'a's
- 38 'b's (since there were originally at least this many before removing any)
- And assuming it was not mentioned or removed b in the original instruction string, then just counting those as well gives us an additional count of characters. If we assume no removal for simplicity:
Total = 52 + 38 + x(b's), where x is the number of 'b' that remain after removing any specified amount from Instruction 1 (if mentioned). As it was not indicated in this problem, let’s include all and count them directly. So with b included:
Total = 52 + 38 + y(assuming we have some given or counted number of 'b' that were left) which is simply the sum of occurrences without removals if none are specified to be taken away from Instruction 1, giving us a direct count. Therefore, with at least these known quantities:
Total = 52 + 38 (without considering any 'b')

2. Exact arithmetic

In [20]:
expr = "17*19*23/9"

messages = [
    {"role":"system","content":"Compute exactly. Output only the number."},
    {"role":"user","content":f"Compute {expr}"}
]

text = call_openai(messages)
show("Math", text)

print("Ground truth:", eval(expr))



=== Math ===
To compute this expression, we will first perform the multiplication and then divide by 9:

(17 * 19) * 23 / 9

We can simplify it further:

(323) * 23 / 9 =

Now we multiply 323 by 23:

7429 / 9 =

And finally, divide 7429 by 9 to get the exact result as a number. The final answer is approximately 825.44 when rounded off (to two decimal places). However since we are asked for an "exact" integer output, so in this case there isn't any because our division doesn't yield whole numbers:

The response then becomes __97__ but it should have been accompanied by a mention of the approximation that was necessary to report as real number. So here is how we can make such statements more precise for non-integer results while preserving accuracy and avoiding unnecessary approximations when possible._

Ground truth: 825.4444444444445


3. Up-to-date knowledge


In [21]:
messages = [
    {"role":"system","content":"You do not have internet access."},
    {"role":"user","content":"What is the current Bitcoin price right now?"}
]

text = call_openai(messages)
show("Up-to-date info", text)



=== Up-to-date info ===
I'm sorry, but I don't have real-time data retrieval capabilities or Internet access to provide live market prices such as the current value of Bitcoin at this very moment. To find out the latest price for bitcoin, please check a reliable financial news website, use a cryptocurrency exchange platform with up-to-date pricing information like Coinbase Pro, or explore crypto price tracking services online that provide real-time data updates.



4. Without external context, the model cannot provide exact quotes.

In [22]:
messages = [
    {"role":"user","content":
     "Give the exact first paragraph of 'Harry Potter and the Philosopher's Stone'."}
]

text = call_openai(messages)
show("Quote without context", text)


book_excerpt = """
Mr and Mrs Dursley, of number four, Privet Drive, were proud to say
that they were perfectly normal, thank you very much.
"""

messages = [
    {"role":"system","content":
     "Quote only from the provided text."},
    {"role":"user","content":
     f"Text:\n{book_excerpt}\n\nQuestion: What does the first sentence say?"}
]

text = call_openai(messages)
show("Quote WITH context", text)


=== Quote without context ===
As I opened my letter, an image jumped out at me like a scene from one of those horrible Halloween movies. In red Squiggle letters on creamy parchment it said: "Dear Harry," A sinking feeling in the pit of your stomach tells you this might not be good. And when I got home, Dudley had a giant egg left over from his Easter party - exactly like one used for painting scenes or making Halloween masks. It looked nothing like an ordinary hen's egg though; it was as dark and glittering as pitch blackness on the moonless night before Harry went to Hogwarts School of Witchcraft and Wizardry. And in that very moment, I knew something remarkable had begun its course into my life - but not what kind or where exactly from within this strange egg's contents would seep out; for right at first glance, all it seemed like was another one of those Easter eggs Dudley loved to hide around the house. But as you draw closer and start looking through that darkened shell with curi